In [1]:
import pandas as pd
df = pd.DataFrame({'dob': ['1995-03-15', '1988-11-02', '2001-07-20']})
df['dob'] = pd.to_datetime(df['dob'])
df['age'] = 2024 - df['dob'].dt.year
df['birth_month'] = df['dob'].dt.month
df['birth_day'] = df['dob'].dt.day_name()
bins = [0, 28, 44, 60, 100]
labels = ['Gen-Z', 'Millennial', 'Gen-X', 'Boomer']
df['generation'] = pd.cut(df['age'], bins=bins, labels=labels)
print(df[['age', 'birth_month', 'birth_day', 'generation']])

   age  birth_month  birth_day  generation
0   29            3  Wednesday  Millennial
1   36           11  Wednesday  Millennial
2   23            7     Friday       Gen-Z


In [2]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
df['purchased'] = ['Yes', 'No', 'Yes']
df['size'] = ['Small', 'Medium', 'Large']
le = LabelEncoder()
df['purchased_enc'] = le.fit_transform(df['purchased'])
size_order = {'Small': 0, 'Medium': 1, 'Large': 2}
df['size_enc'] = df['size'].map(size_order)
print(df[['purchased', 'purchased_enc', 'size', 'size_enc']])

  purchased  purchased_enc    size  size_enc
0       Yes              1   Small         0
1        No              0  Medium         1
2       Yes              1   Large         2


In [3]:
import pandas as pd
df = pd.DataFrame({'city': ['Delhi', 'Mumbai', 'Chennai', 'Delhi']})
df_encoded = pd.get_dummies(df, columns=['city'], dtype=int)
print(df_encoded)

print("-----------------------------------------------")
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded = ohe.fit_transform(df[['city']])

print(encoded)

   city_Chennai  city_Delhi  city_Mumbai
0             0           1            0
1             0           0            1
2             1           0            0
3             0           1            0
-----------------------------------------------
[[1. 0.]
 [0. 1.]
 [0. 0.]
 [1. 0.]]


In [4]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'feature_A': [1, 2, np.nan, 4, 5],
    'feature_B': ['X', 'Y', 'Z', np.nan, 'Y'],
    'feature_C': [10.5, np.nan, 12.3, 14.0, np.nan]
})

print('Missing values count per column:')
print(df.isnull().sum())
print('\n')
print('Percentage missing per column:')
print(df.isnull().mean() * 100)

Missing values count per column:
feature_A    1
feature_B    1
feature_C    2
dtype: int64


Percentage missing per column:
feature_A    20.0
feature_B    20.0
feature_C    40.0
dtype: float64


In [5]:
from sklearn.impute import SimpleImputer
import numpy as np
numeric_cols = ['feature_A', 'feature_C']
num_imputer = SimpleImputer(strategy='median')

df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

categorical_cols = ['feature_B']
cat_imputer = SimpleImputer(strategy='most_frequent')

df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])


print('Total missing values after imputation:', df.isnull().sum().sum())
print('\nDataFrame after imputation:')
print(df)

Total missing values after imputation: 0

DataFrame after imputation:
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         Y       14.0
4        5.0         Y       12.3


In [6]:
Q1 = df['feature_A'].quantile(0.25)
Q3 = df['feature_A'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df_clean_iqr = df[(df['feature_A'] >= lower) & (df['feature_A'] <= upper)]
print("DataFrame after IQR outlier removal on 'feature_A':")
print(df_clean_iqr)
print('\n')

from scipy import stats
z_scores = stats.zscore(df['feature_A'])
df_clean_zscore = df[abs(z_scores) < 3]
print("DataFrame after Z-score outlier removal on 'feature_A' (threshold 3):")
print(df_clean_zscore)

DataFrame after IQR outlier removal on 'feature_A':
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         Y       14.0
4        5.0         Y       12.3


DataFrame after Z-score outlier removal on 'feature_A' (threshold 3):
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         Y       14.0
4        5.0         Y       12.3


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

X = df[['feature_A', 'feature_C']]
y = (df['feature_A'] > df['feature_A'].median()).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42)),
])

pipe.fit(X_train, y_train)

print('Accuracy:', pipe.score(X_test, y_test))

X_new = pd.DataFrame({
    'feature_A': [6.0, np.nan, 8.0],
    'feature_C': [15.0, 16.0, np.nan]
})
predictions = pipe.predict(X_new)
print('Predictions for new data:', predictions)

Accuracy: 1.0
Predictions for new data: [1 1 1]


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
np.random.seed(42)
n = 500
df = pd.DataFrame({
'age': np.random.randint(22, 60, n).astype(float),
'salary': np.random.randint(25000, 120000, n).astype(float),
'dept': np.random.choice(['Eng','Sales','HR','Finance'], n),
'education': np.random.choice(['HS','BSc','MSc','PhD'], n),
'years_exp': np.random.randint(0, 35, n).astype(float),
'left_job': np.random.randint(0, 2, n),
})
for col in ['age', 'salary', 'dept']:
    df.loc[df.sample(frac=0.12).index, col] = np.nan
df.loc[0, 'salary'] = 999999
df.loc[1, 'age'] = 150
print('Shape:', df.shape)
print(df.isnull().sum())

In [ ]:
df_base = df.dropna()
X_base = pd.get_dummies(df_base.drop('left_job', axis=1))
y_base = df_base['left_job']
X_tr, X_te, y_tr, y_te = train_test_split(X_base, y_base,
test_size=0.2, random_state=42)
baseline = LogisticRegression(max_iter=200)
baseline.fit(X_tr, y_tr)
base_acc = accuracy_score(y_te, baseline.predict(X_te))
print(f'Baseline accuracy: {base_acc:.3f}')

Baseline accuracy: 0.464


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
num_cols = ['age', 'salary', 'years_exp']
cat_cols = ['dept', 'education']
df['age'] = df['age'].clip(upper=100)
df['salary'] = df['salary'].clip(upper=200000)
X = df.drop('left_job', axis=1)
y = df['left_job']
X_tr, X_te, y_tr, y_te = train_test_split(X, y,
test_size=0.2, random_state=42)
num_pipe = Pipeline([
('impute', SimpleImputer(strategy='median')),
('scale', StandardScaler()),
])
cat_pipe = Pipeline([
('impute', SimpleImputer(strategy='most_frequent')),
('encode', OneHotEncoder(handle_unknown='ignore',
sparse_output=False)),
])
preprocessor = ColumnTransformer([
('num', num_pipe, num_cols),
('cat', cat_pipe, cat_cols),
])
model = Pipeline([
('prep', preprocessor),
('clf', LogisticRegression(max_iter=1000)),
])
model.fit(X_tr, y_tr)
clean_acc = accuracy_score(y_te, model.predict(X_te))
print(f'Clean pipeline accuracy: {clean_acc:.3f}')
print(f'Improvement: +{clean_acc - base_acc:.3f}')

Clean pipeline accuracy: 0.460
Improvement: +-0.004


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(X_te)
print(classification_report(y_te, y_pred,
target_names=['Stayed', 'Left']))
print(confusion_matrix(y_te, y_pred))
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')

              precision    recall  f1-score   support

      Stayed       0.42      0.33      0.37        48
        Left       0.48      0.58      0.53        52

    accuracy                           0.46       100
   macro avg       0.45      0.46      0.45       100
weighted avg       0.45      0.46      0.45       100

[[16 32]
 [22 30]]
CV accuracy: 0.432 +/- 0.070
